# Die Transformer-Architektur — Interaktives Lernnotebook

Dieses Notebook erklärt die wichtigsten Bausteine moderner KI-Modelle wie **GPT** und **BERT**.
Alle Code-Zellen können direkt ausgeführt und verändert werden!

---

## Überblick: Was ist ein Transformer?

Ein **Transformer** ist eine neuronale Netzwerk-Architektur, die 2017 von Google in dem Paper
*"Attention is All You Need"* vorgestellt wurde.

Kernidee: Statt Sätze Wort für Wort zu verarbeiten (wie RNNs), schaut der Transformer
**alle Wörter gleichzeitig an** und lernt, welche Wörter füreinander wichtig sind.

### Die 4 Hauptbausteine:
1.  **Tokenisierung** — Text in Zahlen umwandeln
2.  **Embeddings & Positional Encoding** — Bedeutung und Position kodieren
3.  **Self-Attention** — Beziehungen zwischen Wörtern verstehen
4.  **Feed-Forward Netzwerk** — Informationen verarbeiten

In [ ]:
# Benötigte Bibliotheken installieren und importieren
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Für schönere Plots
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12

print("✅ Bibliotheken geladen! Bereit zum Loslegen.")

---
## 1. 📌 Tokenisierung — Text zu Zahlen

Computer verstehen keinen Text. Deshalb wird jedes Wort (oder Wortteil) in eine **Zahl** (Token-ID) umgewandelt.

**Beispiel:** `"Der Hund schläft"` → `[4, 27, 156]`

In [ ]:
# Einfacher eigener Tokenizer (Wort-Level)

class EinfacherTokenizer:
    def __init__(self, texte):
        # Vokabular aus allen Wörtern aufbauen
        woerter = set()
        for text in texte:
            for wort in text.lower().split():
                woerter.add(wort)
        self.wort_zu_id = {wort: i+1 for i, wort in enumerate(sorted(woerter))}
        self.id_zu_wort = {i: w for w, i in self.wort_zu_id.items()}
        self.wort_zu_id['[UNK]'] = 0  # Unbekannte Wörter

    def kodieren(self, text):
        return [self.wort_zu_id.get(w, 0) for w in text.lower().split()]

    def dekodieren(self, ids):
        return ' '.join([self.id_zu_wort.get(i, '[UNK]') for i in ids])


# ✏️ HIER KANNST DU DEINEN EIGENEN TEXT EINGEBEN!
trainings_texte = [
    "der hund schläft",
    "die katze jagt den hund",
    "der vogel singt ein lied",
]

tokenizer = EinfacherTokenizer(trainings_texte)

# ✏️ PROBIERE VERSCHIEDENE SÄTZE AUS!
mein_satz = "der hund jagt die katze"

token_ids = tokenizer.kodieren(mein_satz)
print(f"Eingabe:    '{mein_satz}'")
print(f"Token-IDs:   {token_ids}")
print(f"Zurück:     '{tokenizer.dekodieren(token_ids)}'")
print(f"\nVokabular-Größe: {len(tokenizer.wort_zu_id)} Wörter")
print(f"Vokabular: {sorted(tokenizer.wort_zu_id.keys())}")

---
## 2. 🗺️ Embeddings & Positional Encoding

### Word Embeddings
Jede Token-ID wird in einen **Vektor** (eine Liste von Zahlen) umgewandelt.
Ähnliche Wörter haben ähnliche Vektoren — der Transformer lernt diese Vektoren selbst!

### Positional Encoding
Da der Transformer alle Wörter gleichzeitig verarbeitet, muss er auch wissen,
an welcher **Position** jedes Wort steht. Das Positional Encoding fügt diese Information hinzu.

Die Formel nutzt Sinus- und Kosinusfunktionen:
$$PE(pos, 2i) = \sin\left(\frac{pos}{10000^{2i/d}}\right)$$
$$PE(pos, 2i+1) = \cos\left(\frac{pos}{10000^{2i/d}}\right)$$

In [ ]:
def positional_encoding(max_laenge, d_model):
    """Berechnet das Positional Encoding für alle Positionen."""
    PE = np.zeros((max_laenge, d_model))
    for pos in range(max_laenge):
        for i in range(0, d_model, 2):
            PE[pos, i]   = np.sin(pos / (10000 ** (i / d_model)))
            if i+1 < d_model:
                PE[pos, i+1] = np.cos(pos / (10000 ** (i / d_model)))
    return PE


# ✏️ EXPERIMENTIERE MIT DIESEN WERTEN!
MAX_LAENGE = 20   # Maximale Satzlänge
D_MODEL    = 64   # Größe des Embedding-Vektors (probiere 16, 32, 128)

pe = positional_encoding(MAX_LAENGE, D_MODEL)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap des Positional Encoding
im = axes[0].imshow(pe, aspect='auto', cmap='RdYlBu')
axes[0].set_xlabel('Embedding-Dimension')
axes[0].set_ylabel('Position im Satz')
axes[0].set_title('Positional Encoding (Heatmap)')
plt.colorbar(im, ax=axes[0])

# Einzelne Dimensionen als Kurven
for dim in [0, 1, 4, 5, 10, 11]:
    axes[1].plot(pe[:, dim], label=f'Dim {dim}')
axes[1].set_xlabel('Position im Satz')
axes[1].set_ylabel('Wert')
axes[1].set_title('Positional Encoding (ausgewählte Dimensionen)')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()
print(f"\nJeder Token bekommt einen Vektor der Größe {D_MODEL}.")
print(f"Jede Position hat ein einzigartiges Muster aus Sin/Cos-Wellen.")

---
## 3. 👁️ Self-Attention — Das Herzstück des Transformers

Self-Attention beantwortet die Frage: **"Welche anderen Wörter sind für dieses Wort gerade wichtig?"**

### Wie es funktioniert:
Jedes Wort wird in **3 Vektoren** umgewandelt:
- **Q** (Query)  — "Wonach suche ich?"
- **K** (Key)    — "Was habe ich anzubieten?"
- **V** (Value)  — "Was ist mein Inhalt?"

Die **Attention-Gewichte** werden berechnet als:
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

Hohe Gewichte = "Schau genau auf dieses Wort!"

In [ ]:
def softmax(x, axis=-1):
    e_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e_x / e_x.sum(axis=axis, keepdims=True)

def self_attention(Q, K, V):
    """Berechnet Self-Attention."""
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)    # Ähnlichkeiten berechnen
    gewichte = softmax(scores)           # In Wahrscheinlichkeiten umwandeln
    ausgabe = gewichte @ V               # Gewichtete Summe der Values
    return ausgabe, gewichte


# ✏️ PROBIERE VERSCHIEDENE SÄTZE AUS!
satz = "der hund schläft"
woerter = satz.split()
n = len(woerter)

# Zufällige (aber reproduzierbare) Q, K, V Matrizen
np.random.seed(42)
d_k = 8  # ✏️ Dimension der Q/K/V Vektoren
Q = np.random.randn(n, d_k)
K = np.random.randn(n, d_k)
V = np.random.randn(n, d_k)

ausgabe, gewichte = self_attention(Q, K, V)

# Visualisierung der Attention-Gewichte
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

im = axes[0].imshow(gewichte, cmap='Blues', vmin=0, vmax=1)
axes[0].set_xticks(range(n))
axes[0].set_yticks(range(n))
axes[0].set_xticklabels(woerter, fontsize=13)
axes[0].set_yticklabels(woerter, fontsize=13)
axes[0].set_xlabel('Keys (Schlüssel)')
axes[0].set_ylabel('Queries (Anfragen)')
axes[0].set_title('Attention-Gewichte\n(Zeile lesen: "Worauf schaut dieses Wort?")')
for i in range(n):
    for j in range(n):
        axes[0].text(j, i, f'{gewichte[i,j]:.2f}', ha='center', va='center',
                    color='white' if gewichte[i,j] > 0.5 else 'black', fontsize=11)
plt.colorbar(im, ax=axes[0])

# Balkendiagramm für das erste Wort
axes[1].bar(woerter, gewichte[0], color=['#2196F3' if i == 0 else '#90CAF9' for i in range(n)])
axes[1].set_title(f'Attention von "{woerter[0]}" auf alle Wörter')
axes[1].set_ylabel('Attention-Gewicht')
axes[1].set_ylim(0, 1)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()
print("\n💡 Jede Zeile zeigt, wie viel Aufmerksamkeit ein Wort auf die anderen Wörter lenkt.")
print("   Die Gewichte in jeder Zeile summieren sich zu 1.0")

---
## 3b. 👁️👁️ Multi-Head Attention

In echten Transformers gibt es **mehrere Attention-Köpfe** gleichzeitig.
Jeder Kopf kann auf **verschiedene Aspekte** achten:
- Kopf 1 → Grammatik (Subjekt-Verb)
- Kopf 2 → Bedeutung (Synonyme)
- Kopf 3 → Kontext (Bezugswörter)
- ...

Die Ausgaben aller Köpfe werden am Ende zusammengeführt.

In [ ]:
# ✏️ EXPERIMENTIERE: Wie verändert sich die Attention mit mehr Köpfen?
ANZAHL_KOEPFE = 4  # Probiere 1, 2, 4, 8

satz = "die katze jagt den hund"
woerter = satz.split()
n = len(woerter)

fig, axes = plt.subplots(1, ANZAHL_KOEPFE, figsize=(4 * ANZAHL_KOEPFE, 4))
if ANZAHL_KOEPFE == 1:
    axes = [axes]

alle_gewichte = []
for kopf in range(ANZAHL_KOEPFE):
    np.random.seed(kopf * 10)  # Jeder Kopf hat andere Gewichte
    Q = np.random.randn(n, 8)
    K = np.random.randn(n, 8)
    V = np.random.randn(n, 8)
    _, gew = self_attention(Q, K, V)
    alle_gewichte.append(gew)

    im = axes[kopf].imshow(gew, cmap='YlOrRd', vmin=0, vmax=1)
    axes[kopf].set_xticks(range(n))
    axes[kopf].set_yticks(range(n))
    axes[kopf].set_xticklabels(woerter, rotation=45, ha='right', fontsize=9)
    axes[kopf].set_yticklabels(woerter, fontsize=9)
    axes[kopf].set_title(f'Kopf {kopf+1}')

plt.suptitle('Multi-Head Attention: Jeder Kopf "schaut" anders!', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print(f"\n{ANZAHL_KOEPFE} Köpfe × {n} Wörter = {ANZAHL_KOEPFE * n} verschiedene Attention-Perspektiven!")

---
## 4. 🧠 Feed-Forward Netzwerk

Nach der Attention-Schicht kommt ein einfaches **neuronales Netzwerk** (2 Schichten).
Es verarbeitet jedes Wort **unabhängig** und fügt Nicht-Linearität hinzu.

$$\text{FFN}(x) = \text{ReLU}(xW_1 + b_1)W_2 + b_2$$

**ReLU** ist die Aktivierungsfunktion: $\text{ReLU}(x) = \max(0, x)$

In [ ]:
def relu(x):
    return np.maximum(0, x)

def feed_forward(x, d_model, d_ff):
    """Einfaches 2-schichtiges Feed-Forward Netzwerk."""
    np.random.seed(7)
    W1 = np.random.randn(d_model, d_ff) * 0.1
    b1 = np.zeros(d_ff)
    W2 = np.random.randn(d_ff, d_model) * 0.1
    b2 = np.zeros(d_model)

    hidden = relu(x @ W1 + b1)   # Schicht 1 + ReLU
    return hidden @ W2 + b2        # Schicht 2


# ✏️ EXPERIMENTIERE MIT DER NETZWERKGRÖSSE!
D_MODEL = 16   # Embedding-Größe
D_FF    = 64   # Innere Schicht (meist 4× D_MODEL)

# Beispiel-Eingabe (3 Wörter, je ein D_MODEL-Vektor)
x = np.random.randn(3, D_MODEL)
ausgabe = feed_forward(x, D_MODEL, D_FF)

# ReLU visualisieren
x_vals = np.linspace(-3, 3, 100)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(x_vals, relu(x_vals), color='#E53935', linewidth=2.5, label='ReLU(x)')
axes[0].plot(x_vals, x_vals, '--', color='gray', alpha=0.5, label='x (linear)')
axes[0].axhline(0, color='black', linewidth=0.5)
axes[0].axvline(0, color='black', linewidth=0.5)
axes[0].set_title('ReLU Aktivierungsfunktion')
axes[0].set_xlabel('Eingabe x')
axes[0].set_ylabel('ReLU(x)')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].bar(range(D_MODEL), x[0], alpha=0.6, label='Eingabe', color='steelblue')
axes[1].bar(range(D_MODEL), ausgabe[0], alpha=0.6, label='Ausgabe FFN', color='tomato')
axes[1].set_title('Feed-Forward: Eingabe vs. Ausgabe (Wort 1)')
axes[1].set_xlabel('Dimension')
axes[1].set_ylabel('Aktivierung')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()
print(f"\nFFN-Architektur: {D_MODEL} → {D_FF} → {D_MODEL} Neuronen")
print(f"Anzahl Parameter: {D_MODEL*D_FF + D_FF} (Schicht 1) + {D_FF*D_MODEL + D_MODEL} (Schicht 2) = {2*(D_MODEL*D_FF + D_FF) + D_MODEL} gesamt")

---
## 5. 🔗 Alles zusammen — Ein Transformer-Block

Ein vollständiger Transformer-Block kombiniert:
1. **Self-Attention** (mit Skip-Connection)
2. **Layer Normalization**
3. **Feed-Forward Netzwerk** (mit Skip-Connection)
4. **Layer Normalization**

In echten Modellen werden viele solcher Blöcke **gestapelt** (GPT-3: 96 Blöcke!).

In [ ]:
def layer_norm(x, eps=1e-6):
    """Layer Normalization: stabilisiert das Training."""
    mu = x.mean(axis=-1, keepdims=True)
    sigma = x.std(axis=-1, keepdims=True)
    return (x - mu) / (sigma + eps)

def transformer_block(x, d_ff=32):
    """Ein vollständiger Transformer-Block."""
    n, d_model = x.shape

    # 1. Multi-Head Self-Attention
    np.random.seed(1)
    Q = x @ np.random.randn(d_model, d_model) * 0.1
    K = x @ np.random.randn(d_model, d_model) * 0.1
    V = x @ np.random.randn(d_model, d_model) * 0.1
    attn_ausgabe, gewichte = self_attention(Q, K, V)

    # 2. Skip-Connection + Layer Norm
    x = layer_norm(x + attn_ausgabe)

    # 3. Feed-Forward
    ffn_ausgabe = feed_forward(x, d_model, d_ff)

    # 4. Skip-Connection + Layer Norm
    x = layer_norm(x + ffn_ausgabe)

    return x, gewichte


# ✏️ STELLE EIN: Wie viele Transformer-Blöcke?
ANZAHL_BLOECKE = 3
D_MODEL = 16

satz = "die katze schläft"
n = len(satz.split())

np.random.seed(0)
x = np.random.randn(n, D_MODEL)   # Simulierte Embeddings

fig, axes = plt.subplots(1, ANZAHL_BLOECKE, figsize=(5 * ANZAHL_BLOECKE, 4))
if ANZAHL_BLOECKE == 1:
    axes = [axes]

woerter = satz.split()
for block_nr in range(ANZAHL_BLOECKE):
    x, gew = transformer_block(x)
    im = axes[block_nr].imshow(gew, cmap='Greens', vmin=0, vmax=1)
    axes[block_nr].set_xticks(range(n))
    axes[block_nr].set_yticks(range(n))
    axes[block_nr].set_xticklabels(woerter, fontsize=11)
    axes[block_nr].set_yticklabels(woerter, fontsize=11)
    axes[block_nr].set_title(f'Block {block_nr+1} — Attention')
    plt.colorbar(im, ax=axes[block_nr])

plt.suptitle(f'{ANZAHL_BLOECKE} gestapelte Transformer-Blöcke', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print("\n💡 Mit jedem Block lernt das Modell abstraktere Repräsentationen!")
print(f"   Ausgabe-Form: {x.shape} (3 Wörter × {D_MODEL} Dimensionen)")

---
## 📊 Zusammenfassung: Transformer vs. andere Architekturen

| Merkmal | RNN/LSTM | Transformer |
|---------|----------|-------------|
| Verarbeitung | Sequenziell (Wort für Wort) | Parallel (alle gleichzeitig) |
| Kontext | Begrenzt (Vergessen) | Unbegrenzt (Attention) |
| Training | Langsam | Schnell (GPU-freundlich) |
| Skalierung | Schwierig | Sehr gut (GPT-4: ~1 Billion Parameter) |

---

## 🚀 Weiterführende Aufgaben

1. **Aufgabe 1**: Ändere `ANZAHL_KOEPFE` in Zelle 3b auf 1 und 8. Was fällt dir auf?
2. **Aufgabe 2**: Verändere `D_MODEL` und `D_FF` im FFN. Wie ändert sich die Parameterzahl?
3. **Aufgabe 3**: Füge ein eigenes Wort zum Tokenizer hinzu. Was passiert mit unbekannten Wörtern `[UNK]`?
4. **Aufgabe 4**: Warum teilt man in der Attention-Formel durch $\sqrt{d_k}$? (Entferne die Division und beobachte die Softmax-Ausgabe!)

---
*Notebook erstellt für den KI-Unterricht | Basiert auf "Attention is All You Need" (Vaswani et al., 2017)*